# 01 — Ingest PubMedQA into ArangoDB (run once)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vardhjain/graphrag-pubmedqa-ablation/blob/main/notebooks/01_ingest.ipynb)

Thin Colab wrapper around `scripts/ingest.py`. Builds the **leakage-free** knowledge graph
(Papers / Chunks / Concepts + HAS_CONTEXT / MENTIONS edges).

**Before running**, add these to the Colab **Secrets** panel (key icon, left sidebar):
- `ARANGO_PASS` — your ArangoDB password (required)
- `ARANGO_HOST` — your endpoint, e.g. `https://<your-deployment>.arangodb.cloud:8529`
  (ArangoDB Oasis offers a free tier; or run any reachable ArangoDB)

A **GPU** runtime (+ High-RAM) speeds the embedding pass. Run this notebook **once**,
then use `02_benchmark.ipynb`. Ingesting labeled + unlabeled (~62k papers) is mostly
network-bound on the inserts.

In [ ]:
# Reset-safe clone: always starts from /content and removes any prior copy,
# so re-running this cell can never nest a second checkout.
%cd /content
!rm -rf graphrag-pubmedqa-ablation
!git clone -b main https://github.com/vardhjain/graphrag-pubmedqa-ablation.git -q
%cd graphrag-pubmedqa-ablation
!pip install -q -r requirements.txt

In [ ]:
import os
from google.colab import userdata

# Pull connection settings from Colab Secrets (nothing is hardcoded).
for key in ['ARANGO_PASS', 'ARANGO_HOST', 'ARANGO_DB']:
    try:
        val = userdata.get(key)
        if val:
            os.environ[key] = val
    except Exception:
        pass

assert os.environ.get('ARANGO_PASS'), 'Add ARANGO_PASS in the Secrets panel.'
print('ARANGO_HOST:', os.environ.get('ARANGO_HOST', '(default http://localhost:8529)'))

In [ ]:
# Quick smoke test first (labeled split only, ~1k papers) to confirm the
# connection + schema before the full run:
!python scripts/ingest.py --no-unlabeled

In [ ]:
# Full ingestion (labeled + unlabeled). Safe to re-run: papers/chunks upsert by key.
!python scripts/ingest.py